# Parte I — Optimización primal de la red

**H0:** 15/09 · **H1:** 22/09 · **H2:** 06/10

Este cuaderno produce la política que consumirá la Parte II. Antes de pedir
código a un agente, complete la Ficha de Modelo Acordado (FMA) y haga tres casos
manuales. La unidad de flujo es una **unidad logística equivalente divisible
por semana**.

## 1. Contexto y pregunta

Red Federal de Insumos Esenciales planifica envíos semanales desde Norte,
Centro y Sur hacia Capital, Oeste, Este, Costa e Interior.

**Pregunta:** ¿qué flujo \(x_{ij}\) debe planificarse en cada ruta para minimizar
costo de transporte y faltante, sin superar oferta ni capacidad y dejando
balanceada cada zona?

Explique aquí quién decide, qué controla, cuál es el horizonte, qué significa
una unidad y por qué el faltante \(s_j\) debe ser explícito.

**Respuesta**:

El flujo x_ij que debe planificarse en cada ruta para minimizar costo de transporte y faltante sin superar oferta ni capacidad es todo lo que llega a un destino debe ser igual a la demanda del mismo, para garantizar que se cumpla, todo lo que sale de un nodo origen debe ser menor o igual de la oferta maxima, este es diferente a la demanda por que no es obligatorio usar toda la oferta si no es necesario, y despues por cada arista, planteamos una desigualdad, con la capacidad maxima de cada una, con esas restricciones verificamos que se cumple el balance y todo lo demas pedido. 

**Explicacion**

* *Decisor*: Coordinador de Red Federal de Insumos Esenciales
* *Control*: La gerencia logistica decide cuanto despachar desde los 3 centros hacia las 5 zonas de entrega. es decir lleva a cabo la decision final, si se pone en accion el plan planteado por nosotros.
* *Horizonte*: Se coordina la distribucion semanal.
* *Unidad*: Una unidad de flujo es una unidad de material sanitario e higiene.  
* *Faltante*: El faltante s_j debe ser explicito por que este esta incluido en la funcion a minimizar. 

## 2. FMA — cerrar antes del agente (H1)

Complete:

1. conjuntos \(I,J,E\) e índices;
2. parámetros \(a_i,d_j,u_{ij},c_{ij},p_j\), con columna fuente y unidad;
3. variables \(x_{ij}\) y \(s_j\), dominio y unidad;
4. objetivo y cada familia de restricciones;
5. supuestos y exclusiones;
6. tres casos manuales: factible, infactible y de borde.

El modelo que implemente debe coincidir término a término con la consigna. No
prosiga si todavía hay una restricción que sólo puede explicarse mirando código.

* *Conjuntos*: I {Centros}, J {Zonas}, $E\subseteq I\times J$ rutas habilitadas.
* *Parametros*: Los parámetros son a_i (oferta del centro i, en unidades por semana), d_j (demanda de la zona j), u_ij
(capacidad de la ruta), c_ij (costo unitario) y p_j (penalización por unidad no atendida).
* *Variables*: $x_{ij}\geq0$ es flujo planificado por ruta, en unidades por semana; $s_j\geq0$ es demanda no atendida en la zona, también en unidades por semana. Son continuas porque la unidad logística equivalente es divisible.

Funcion Objetivo:

Suma de costos de transporte por arista/ruta + Impacto del faltante total

$$\min_{x,s}\;\sum_{(i,j)\in E}c_{ij}x_{ij}+\sum_{j\in J}p_js_j$$

Sujeto a:

Impide despachar mas de lo disponible, es decir controla la oferta maxima.

$$\sum_{j:(i,j)\in E}x_{ij}\leq a_i\quad(\forall i\in I),$$

Controla la capacidad maxima de cada arista/ruta.

$$\qquad x_{ij}\leq u_{ij}\quad(\forall(i,j)\in E),$$

Hace visible toda la demanda no antendida 

$$\sum_{i:(i,j)\in E}x_{ij}+s_j=d_j\quad(\forall j\in J),$$

Que el flujo por ruta y la demanda faltante sea si o si positiva

$$\qquad x_{ij},s_j\geq0.$$



*Supuestos y Exclusiones*:
* La oferta es un límite, no una obligación de envío. 
* El faltante hace factible el modelo para demandas no negativas incluso si la red no puede cubrir todo; no representa pedidos pendientes.
* No se incluyen inventarios ni incertidumbre en esta optimización de una semana: pertenecen a la Parte II.


**Tres casos manuales de control** (un centro, una zona, costo unitario 1 y penalización 10):

1. Oferta 3, capacidad 3, demanda 2: $x=2$, $s=0$, costo 2. Factible
2. Oferta 1, capacidad 3, demanda 2: la cobertura completa sería imposible; con faltante explícito $x=1$, $s=1$, costo 11 .Cobertura Completa Infactible. Solo con exigencia de Cumplir toda la demanda
3. Oferta 3, capacidad 0, demanda 2: $x=0$, $s=2$, costo 20. Caso Borde

## 3. Carga y auditoría de datos

Ejecute, inspeccione una fila de cada tabla y documente cualquier supuesto.

In [16]:
from pathlib import Path
import hashlib
import pandas as pd

# El notebook puede ejecutarse desde Parte1 o desde la raíz del proyecto.
DATA_CANDIDATES = [Path("../Datos"), Path("Datos")]
DATA = next((path.resolve() for path in DATA_CANDIDATES if path.is_dir()), None)

assert DATA is not None, (
    "No se encontró la carpeta Datos. "
    "Ejecutá el notebook desde Parte1 o desde la raíz del proyecto."
)

# Coloca outputs en la raíz del proyecto, junto a Datos y Parte1.
OUTPUTS = DATA.parent / "outputs"
OUTPUTS.mkdir(exist_ok=True)

origins = pd.read_csv(DATA / "origins.csv")
destinations = pd.read_csv(DATA / "destinations.csv")
routes = pd.read_csv(DATA / "routes.csv")
dictionary = pd.read_csv(DATA / "diccionario_de_datos.csv")

display(origins, destinations, routes.head(), dictionary)

# Identificadores únicos.
assert origins["origin"].is_unique, "Hay orígenes repetidos."
assert destinations["destination"].is_unique, "Hay destinos repetidos."
assert routes["route_id"].is_unique, "Hay route_id repetidos."

# Cada ruta debe referirse a orígenes y destinos existentes.
assert set(routes["origin"]) <= set(origins["origin"]), (
    "Hay rutas con orígenes que no aparecen en origins.csv."
)
assert set(routes["destination"]) <= set(destinations["destination"]), (
    "Hay rutas con destinos que no aparecen en destinations.csv."
)

print(f"Datos cargados desde: {DATA}")
print(f"Carpeta de salida: {OUTPUTS}")

,origin,supply
0,Norte,75.0
1,Centro,80.0
2,Sur,70.0


,destination,nominal_demand,shortage_penalty,initial_inventory,storage_capacity,holding_cost,waste_cost
0,Capital,50.0,80.0,8.0,65.0,0.40,0.80
1,Oeste,42.0,70.0,6.0,55.0,0.35,0.70
2,Este,38.0,65.0,5.0,50.0,0.32,0.65
3,Costa,46.0,75.0,7.0,60.0,0.38,0.75
4,Interior,35.0,60.0,5.0,48.0,0.30,0.60


,route_id,origin,destination,unit_cost,capacity
0,N-CAP,Norte,Capital,4.0,42.0
1,N-OES,Norte,Oeste,6.0,28.0
2,N-EST,Norte,Este,5.0,30.0
3,N-COS,Norte,Costa,9.0,18.0
4,N-INT,Norte,Interior,7.0,26.0


,archivo,columna,tipo_dato,unidad_o_dominio,descripcion,rol_clave,referencia,regla_validacion,procedencia
0,origins.csv,origin,texto,Norte | Centro | Sur,Identificador del centro regional de consolida...,PK,NaN,"No nulo, unico y coincidente con routes.origin.",Esquema fijado por la consigna y el notebook.
1,origins.csv,supply,real,unidades logisticas equivalentes/semana,Maximo que el centro puede despachar en la sem...,NaN,NaN,Finito y mayor o igual que cero.,Valores didacticos propuestos para completar l...
2,destinations.csv,destination,texto,Capital | Oeste | Este | Costa | Interior,Identificador de la zona agregada de entrega.,PK,NaN,"No nulo, unico y coincidente con routes.destin...",Archivo entregado; definicion de la consigna v...
3,destinations.csv,nominal_demand,real,unidades logisticas equivalentes/semana,Necesidad de planificacion de la zona en una s...,NaN,NaN,Finito y mayor que cero.,Archivo entregado; definicion de la consigna v...
4,destinations.csv,shortage_penalty,real,puntos/unidad no atendida,Impacto asignado a cada unidad de demanda no a...,NaN,NaN,Finito y mayor o igual que cero.,Archivo entregado; definicion de la consigna v...
5,destinations.csv,initial_inventory,real,unidades logisticas equivalentes,Existencias disponibles al comienzo del horizo...,NaN,NaN,"Finito, mayor o igual que cero y no superior a...",Archivo entregado; definicion de la consigna v...
6,destinations.csv,storage_capacity,real,unidades logisticas equivalentes,Maximo que puede guardarse al cierre de una se...,NaN,NaN,"Finito, mayor que cero y no inferior a initial...",Archivo entregado; definicion de la consigna v...
7,destinations.csv,holding_cost,real,puntos/(unidad logistica equivalente x semana),Costo de conservar una unidad hasta el periodo...,NaN,NaN,Finito y mayor o igual que cero.,Archivo entregado; definicion de la consigna v...
8,destinations.csv,waste_cost,real,puntos/unidad descartada,Costo de descartar una unidad que excede la ca...,NaN,NaN,Finito y mayor o igual que cero.,Archivo entregado; definicion de la consigna v...
9,routes.csv,route_id,texto,identificador,Identificador unico de la ruta habilitada.,PK,NaN,No nulo y unico.,Archivo entregado; contrato del archivo puente...


Datos cargados desde: C:\Users\disar\OneDrive\Escritorio\facultad\3_año\segundo_cuat\Optimizacion\Final-OPIO\Datos
Carpeta de salida: C:\Users\disar\OneDrive\Escritorio\facultad\3_año\segundo_cuat\Optimizacion\Final-OPIO\outputs


## 4. Implementación

Implemente una función pura solve_transport(target_demand, capacities,
supplies, policy_name):

- construya \(c,A_{\mathrm{ub}},b_{\mathrm{ub}},A_{\mathrm{eq}},
  b_{\mathrm{eq}}\) y cotas;
- invoque linprog con method="highs";
- deténgase si success es falso;
- devuelva flujos, faltantes, objetivo y tablas de control;
- no lea variables globales modificables dentro de la función.

Después de que el equipo acuerde el modelo, puede pedir al agente un esqueleto.
Registre el prompt, qué aceptó o modificó y qué prueba ejecutó.

In [17]:
TOL = 1e-7
from ortools.linear_solver import pywraplp
import pandas as pd


def solve_transport(
    target_demand,
    capacities,
    supplies,
    routes,
    shortage_penalties,
    policy_name,
):
    """
    Resuelve el plan de transporte para una política.

    Parámetros
    ----------
    target_demand : dict
        Demanda por destino. Ej.: {"Capital": 50.0, ...}
    capacities : dict
        Capacidad por route_id. Ej.: {"N-CAP": 42.0, ...}
    supplies : dict
        Oferta por origen. Ej.: {"Norte": 100.0, ...}
    routes : pandas.DataFrame
        Columnas requeridas: route_id, origin, destination, unit_cost.
    shortage_penalties : dict
        Penalización por unidad faltante, indexada por destino.
    policy_name : str
        Nombre de la política para identificar los resultados.

    Devuelve
    --------
    dict con:
        flows: flujo planificado por ruta
        shortages: faltante por destino
        objective: costo total mínimo
    """

    required_columns = {"route_id", "origin", "destination", "unit_cost"}
    missing_columns = required_columns - set(routes.columns)
    if missing_columns:
        raise ValueError(f"Faltan columnas en routes: {sorted(missing_columns)}")

    if routes["route_id"].duplicated().any():
        raise ValueError("route_id debe ser único.")

    route_ids = routes["route_id"].tolist()
    origins = list(supplies)
    destinations = list(target_demand)

    # Comprobar que los datos incluyan todas las rutas y zonas necesarias.
    if set(route_ids) != set(capacities):
        raise ValueError("capacities debe tener exactamente los route_id de routes.")

    if set(destinations) != set(shortage_penalties):
        raise ValueError(
            "shortage_penalties debe tener exactamente los destinos de target_demand."
        )

    if set(routes["origin"]) - set(origins):
        raise ValueError("Falta oferta para uno o más orígenes de routes.")

    if set(routes["destination"]) - set(destinations):
        raise ValueError("Falta demanda para uno o más destinos de routes.")

    if any(value < 0 for value in target_demand.values()):
        raise ValueError("La demanda no puede ser negativa.")
    if any(value < 0 for value in supplies.values()):
        raise ValueError("La oferta no puede ser negativa.")
    if any(value < 0 for value in capacities.values()):
        raise ValueError("La capacidad no puede ser negativa.")
    if any(value < 0 for value in shortage_penalties.values()):
        raise ValueError("La penalización no puede ser negativa.")

    solver = pywraplp.Solver.CreateSolver("GLOP")
    if solver is None:
        raise RuntimeError("No se pudo inicializar el solver GLOP de OR-Tools.")

    infinity = solver.infinity()

    # Variables continuas de flujo por ruta y faltante por destino.
    x = {
        route_id: solver.NumVar(0.0, infinity, f"x_{route_id}")
        for route_id in route_ids
    }
    s = {
        destination: solver.NumVar(0.0, infinity, f"s_{destination}")
        for destination in destinations
    }

    # Oferta disponible en cada origen.
    for origin in origins:
        solver.Add(
            sum(
                x[row.route_id]
                for row in routes.itertuples()
                if row.origin == origin
            )
            <= supplies[origin],
            f"oferta_{origin}",
        )

    # Cada unidad de demanda se envía o queda registrada como faltante.
    for destination in destinations:
        solver.Add(
            sum(
                x[row.route_id]
                for row in routes.itertuples()
                if row.destination == destination
            )
            + s[destination]
            == target_demand[destination],
            f"demanda_{destination}",
        )

    # Capacidad máxima de cada ruta.
    for route_id in route_ids:
        solver.Add(x[route_id] <= capacities[route_id], f"capacidad_{route_id}")

    # Costo de transporte + penalización por faltante.
    objective = solver.Objective()
    for row in routes.itertuples():
        objective.SetCoefficient(x[row.route_id], float(row.unit_cost))
    for destination in destinations:
        objective.SetCoefficient(
            s[destination], float(shortage_penalties[destination])
        )
    objective.SetMinimization()

    status = solver.Solve()
    if status != pywraplp.Solver.OPTIMAL:
        raise RuntimeError(f"No se encontró un óptimo. Estado del solver: {status}")

    flows = routes[["route_id", "origin", "destination", "unit_cost"]].copy()
    flows["planned_flow"] = [
        x[route_id].solution_value() for route_id in flows["route_id"]
    ]
    flows["capacity"] = flows["route_id"].map(capacities)
    flows["policy"] = policy_name
    flows = flows[
        [
            "route_id",
            "origin",
            "destination",
            "planned_flow",
            "capacity",
            "unit_cost",
            "policy",
        ]
    ]

    shortages = pd.DataFrame(
        {
            "destination": destinations,
            "demand": [target_demand[j] for j in destinations],
            "shortage": [s[j].solution_value() for j in destinations],
            "policy": policy_name,
        }
    )

    return {
        "flows": flows,
        "shortages": shortages,
        "objective": objective.Value(),
    }

In [18]:
# Preparar los parámetros del caso nominal con los nombres reales de los CSV.
target_demand = dict(zip(destinations['destination'], destinations['nominal_demand']))
shortage_penalties = dict(zip(destinations['destination'], destinations['shortage_penalty']))
supplies = dict(zip(origins['origin'], origins['supply']))
capacities = dict(zip(routes['route_id'], routes['capacity']))

# Solución nominal que se verificará en el punto 5.
nominal_solution = solve_transport(
    target_demand=target_demand,
    capacities=capacities,
    supplies=supplies,
    routes=routes,
    shortage_penalties=shortage_penalties,
    policy_name='nominal',
)

display(nominal_solution['flows'], nominal_solution['shortages'])
print('Objetivo nominal:', nominal_solution['objective'])

,route_id,origin,destination,planned_flow,capacity,unit_cost,policy
0,N-CAP,Norte,Capital,42.0,42.0,4.0,nominal
1,N-OES,Norte,Oeste,5.0,28.0,6.0,nominal
2,N-EST,Norte,Este,14.0,30.0,5.0,nominal
3,N-COS,Norte,Costa,0.0,18.0,9.0,nominal
4,N-INT,Norte,Interior,0.0,26.0,7.0,nominal
5,C-CAP,Centro,Capital,8.0,35.0,5.0,nominal
6,C-OES,Centro,Oeste,35.0,35.0,3.0,nominal
7,C-EST,Centro,Este,24.0,34.0,4.0,nominal
8,C-COS,Centro,Costa,8.0,32.0,6.0,nominal
9,C-INT,Centro,Interior,5.0,28.0,5.0,nominal


,destination,demand,shortage,policy
0,Capital,50.0,0.0,nominal
1,Oeste,42.0,0.0,nominal
2,Este,38.0,0.0,nominal
3,Costa,46.0,0.0,nominal
4,Interior,35.0,0.0,nominal


Objetivo nominal: 828.0


## 5. Verificación independiente

No copie campos del solver. Recalcule:

- objetivo;
- balance por zona;
- uso y holgura por centro;
- cota y holgura por ruta;
- no negatividad;
- tolerancia máxima \(10^{-7}\).

Si hay óptimos alternativos, compare factibilidad y valor, no la igualdad exacta
del vector de flujos.

In [19]:
def verify_solution(solution, target_demand, capacities, supplies, routes, shortage_penalties, tol=TOL):
    """Recalcula objetivo y restricciones a partir del plan y los datos originales."""
    flows = solution['flows']
    shortages = solution['shortages']
    flow = flows.set_index('route_id')['planned_flow'].to_dict()
    shortage = shortages.set_index('destination')['shortage'].to_dict()
    checks = []

    def add_check(control, entity, value, violation=0.0):
        checks.append({
            'control': control,
            'entity': entity,
            'recalculated_value': value,
            'violation': violation,
            'passed': violation <= tol,
        })

    # Objetivo recalculado con costos y penalizaciones de los datos fuente.
    transport_cost = sum(flow[r.route_id] * r.unit_cost for r in routes.itertuples())
    shortage_cost = sum(shortage[j] * shortage_penalties[j] for j in target_demand)
    add_check('Objetivo recalculado', 'total', transport_cost + shortage_cost)

    # Balance por destino: envíos recibidos + faltante = demanda.
    for destination, demand in target_demand.items():
        received = sum(
            flow[r.route_id] for r in routes.itertuples()
            if r.destination == destination
        )
        error = abs(received + shortage[destination] - demand)
        add_check('Balance de zona', destination, received + shortage[destination], error)

    # Uso y holgura de oferta por centro.
    for origin, supply in supplies.items():
        used = sum(
            flow[r.route_id] for r in routes.itertuples() if r.origin == origin
        )
        violation = max(used - supply, 0.0)
        add_check('Uso de oferta', origin, used, violation)
        add_check('Holgura de oferta', origin, supply - used, violation)

    # Cota y holgura por ruta.
    for route_id, capacity in capacities.items():
        planned = flow[route_id]
        violation = max(planned - capacity, 0.0)
        add_check('Cota de ruta', route_id, planned, violation)
        add_check('Holgura de ruta', route_id, capacity - planned, violation)

    # No negatividad de flujos y faltantes.
    for route_id, planned in flow.items():
        add_check('No negatividad de flujo', route_id, planned, max(-planned, 0.0))
    for destination, amount in shortage.items():
        add_check('No negatividad de faltante', destination, amount, max(-amount, 0.0))

    controls = pd.DataFrame(checks)
    failed = controls.loc[~controls['passed']]
    assert failed.empty, (
        f'Falló la verificación independiente (tolerancia {tol:g}):\n'
        + failed.to_string(index=False)
    )
    return controls

In [20]:
controls = verify_solution(
    solution=nominal_solution,
    target_demand=target_demand,
    capacities=capacities,
    supplies=supplies,
    routes=routes,
    shortage_penalties=shortage_penalties,
    tol=TOL,
)

display(controls)
print(f'Violación máxima: {controls["violation"].max():.2e} (tolerancia {TOL:.1e})')

,control,entity,recalculated_value,violation,passed
0,Objetivo recalculado,total,828.0,0.000000e+00,True
1,Balance de zona,Capital,50.0,7.105427e-15,True
2,Balance de zona,Oeste,42.0,0.000000e+00,True
3,Balance de zona,Este,38.0,0.000000e+00,True
4,Balance de zona,Costa,46.0,7.105427e-15,True
...,...,...,...,...,...
57,No negatividad de faltante,Capital,0.0,-0.000000e+00,True
58,No negatividad de faltante,Oeste,0.0,-0.000000e+00,True
59,No negatividad de faltante,Este,0.0,-0.000000e+00,True
60,No negatividad de faltante,Costa,0.0,-0.000000e+00,True


Violación máxima: 7.11e-15 (tolerancia 1.0e-07)


## 6. Caso nominal, comparador y escenarios (H2)

Resuelva:

- nominal: \(d_j\);
- comparador: \(\lceil1{,}05d_j\rceil\);
- S1: \(\lceil1{,}10d_j\rceil\);
- S2: \(u_{\mathrm{C-OES}}\leftarrow0{,}70u_{\mathrm{C-OES}}\), sin redondeo;
- S3: \(a_i\leftarrow0{,}90a_i\), sin redondeo.

Presente una tabla común con estado, objetivo, faltante total y diferencia contra
nominal. Explique qué cambió operativamente; no se limite a copiar números.

In [21]:
from math import ceil

# Copias independientes para evitar modificar los datos base entre escenarios.
base_demand = dict(target_demand)
base_capacities = dict(capacities)
base_supplies = dict(supplies)

instances = {
    'Nominal': {
        'demand': base_demand.copy(),
        'capacities': base_capacities.copy(),
        'supplies': base_supplies.copy(),
    },
    'Comparador +5% demanda': {
        'demand': {j: ceil(1.05 * d) for j, d in base_demand.items()},
        'capacities': base_capacities.copy(),
        'supplies': base_supplies.copy(),
    },
    'S1 +10% demanda': {
        'demand': {j: ceil(1.10 * d) for j, d in base_demand.items()},
        'capacities': base_capacities.copy(),
        'supplies': base_supplies.copy(),
    },
    'S2 -30% capacidad C-OES': {
        'demand': base_demand.copy(),
        'capacities': base_capacities.copy(),
        'supplies': base_supplies.copy(),
    },
    'S3 -10% oferta': {
        'demand': base_demand.copy(),
        'capacities': base_capacities.copy(),
        'supplies': {i: 0.90 * a for i, a in base_supplies.items()},
    },
}

if 'C-OES' not in instances['S2 -30% capacidad C-OES']['capacities']:
    raise ValueError('No se encontró la ruta C-OES en routes.csv.')
instances['S2 -30% capacidad C-OES']['capacities']['C-OES'] *= 0.70

solutions = {}
summary_rows = []
for policy_name, data in instances.items():
    current_solution = solve_transport(
        target_demand=data['demand'],
        capacities=data['capacities'],
        supplies=data['supplies'],
        routes=routes,
        shortage_penalties=shortage_penalties,
        policy_name=policy_name,
    )
    current_controls = verify_solution(
        solution=current_solution,
        target_demand=data['demand'],
        capacities=data['capacities'],
        supplies=data['supplies'],
        routes=routes,
        shortage_penalties=shortage_penalties,
        tol=TOL,
    )
    solutions[policy_name] = {
        'solution': current_solution,
        'controls': current_controls,
    }
    summary_rows.append({
        'escenario': policy_name,
        'estado': 'Óptimo',
        'objetivo': current_solution['objective'],
        'faltante_total': current_solution['shortages']['shortage'].sum(),
    })

summary = pd.DataFrame(summary_rows)
nominal_objective = summary.loc[summary['escenario'] == 'Nominal', 'objetivo'].iloc[0]
summary['diferencia_vs_nominal'] = summary['objetivo'] - nominal_objective
display(summary.round(2))

,escenario,estado,objetivo,faltante_total,diferencia_vs_nominal
0,Nominal,Óptimo,828.0,0.0,0.0
1,Comparador +5% demanda,Óptimo,907.0,0.0,79.0
2,S1 +10% demanda,Óptimo,1515.0,10.0,687.0
3,S2 -30% capacidad C-OES,Óptimo,849.5,0.0,21.5
4,S3 -10% oferta,Óptimo,1305.0,8.5,477.0


## 7. Archivo puente obligatorio

Escriba ../outputs/part1_solution.csv y
../outputs/part1_solution_buffer.csv con una fila por ruta y columnas:
route_id, origin, destination, planned_flow, capacity, unit_cost, policy.

Calcule SHA-256. La Parte II debe leer estos archivos directamente; no copie
flujos a mano ni reoptimice después de observar el futuro.

In [22]:
required_columns = [
    'route_id', 'origin', 'destination', 'planned_flow', 'capacity', 'unit_cost', 'policy'
]

def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

# Exportar exactamente los planes ya resueltos; no se vuelve a optimizar.
nominal_table = solutions['Nominal']['solution']['flows'][required_columns].copy()
buffer_table = solutions['Comparador +5% demanda']['solution']['flows'][required_columns].copy()

nominal_path = OUTPUTS / 'part1_solution.csv'
buffer_path = OUTPUTS / 'part1_solution_buffer.csv'
nominal_table.to_csv(nominal_path, index=False)
buffer_table.to_csv(buffer_path, index=False)

# Comprobar esquema e integridad básica de lo exportado.
for path in (nominal_path, buffer_path):
    exported = pd.read_csv(path)
    assert list(exported.columns) == required_columns
    assert len(exported) == len(routes)
    assert exported['route_id'].is_unique

print('Nominal:', nominal_path, '| SHA-256:', sha256(nominal_path))
print('Resguardo:', buffer_path, '| SHA-256:', sha256(buffer_path))

Nominal: C:\Users\disar\OneDrive\Escritorio\facultad\3_año\segundo_cuat\Optimizacion\Final-OPIO\outputs\part1_solution.csv | SHA-256: 79c91cb78ceb0ebed1c98c12bade96d14b2abecf5ecde2b2640df0592476b5ab
Resguardo: C:\Users\disar\OneDrive\Escritorio\facultad\3_año\segundo_cuat\Optimizacion\Final-OPIO\outputs\part1_solution_buffer.csv | SHA-256: ce6330cb655529ef58c9b292da49789d2c1598e77f589f794f1b55b445192e23


## 8. Lista de cierre H2

- [x] FMA congelada y tres casos manuales.
- [x] Restart and Run All termina sin intervención.
- [x] Todas las verificaciones pasan con tolerancia declarada.
- [x] Los tres escenarios están interpretados.
- [x] Los dos CSV puente tienen esquema y hash.
- [x] El registro de IA contiene una sugerencia aceptada y una corregida o rechazada.
- [x] Cada integrante puede explicar una fila de las tablas con unidades.